# Pipeline Thu Thập Dữ Liệu BĐS TP.HCM Với Scrapling (Production)

**Notebook gọi trực tiếp `crawl_pipeline()` trong `scrapling_crawler.py`** với đầy đủ các tiêu chuẩn kiến trúc:
- **Incremental**: lưu trạng thái `state.json`, bỏ qua tin đã cào, dừng sớm khi gặp liên tiếp nhiều tin cũ (mặc định 15, tránh bẫy tin VIP ghim đầu trang).
- **Apache Parquet Hive-style**: tự động lưu vào `data_1/Raw/year=YYYY/month=MM/day=DD/listings_YYYYMMDD_HHMMSS.parquet` (nén snappy).
- **Fault-tolerant**: retry exponential backoff + jitter (lỗi mạng, HTTP 403/429/5xx), timeout 20s, cô lập lỗi từng tin/từng trang, checkpoint định kỳ, `Ctrl+C`/lỗi bất ngờ vẫn lưu dữ liệu an toàn.
- **Anti-blocking**: độ trễ ngẫu nhiên + giả lập TLS Chrome 124 vượt Cloudflare 403.
- **Logging chuẩn** (`crawler.log`) và báo cáo tổng kết cuối phiên.
- Trích xuất metadata (`Ngày đăng`, `Loại tin`, `Mã tin`) và điền khuyết bằng regex từ mô tả.

> Có thể chạy lại cell cào nhiều lần: chỉ tin mới được thêm vào dataset.

## 0. Cài đặt thư viện chống chặn Cloudflare 403 (Chạy cell này nếu môi trường thiếu thư viện)
Lệnh `%pip` sẽ tự động cài trực tiếp vào đúng Kernel Python của notebook hiện tại mà không cần mở terminal.

In [ ]:
# Cài đặt curl-cffi, patchright và boto3 cho Backblaze B2
%pip install -q "curl-cffi>=0.7.0" patchright boto3


## 1. Khai báo thư viện

In [ ]:
import importlib

import numpy as np
import pandas as pd

import scrapling_crawler
import storage_b2
importlib.reload(scrapling_crawler)  # nạp lại code crawler mới nhất
importlib.reload(storage_b2)        # nạp lại code B2 Storage mới nhất
from scrapling_crawler import crawl_pipeline, fill_specs_with_desc
from storage_b2 import get_b2_manager, verify_b2_connection_and_list


## 2. Cấu hình
Giá trị mặc định đồng bộ với CLI (`python scrapling_crawler.py`), tự động tổ chức Parquet phân vùng Hive theo ngày chạy.

In [ ]:
TARGET_SAMPLES = 3           # số tin MỚI muốn cào trong lần test này (tăng lên 20-50 khi chạy thật)
OUTPUT_DIR = "data_1/Raw"    # nơi lưu phân vùng Hive year=YYYY/month=MM/day=DD/*.parquet
STATE_FILE = "state.json"    # trạng thái các tin đã cào
LOG_FILE = "crawler.log"
EARLY_STOP_THRESHOLD = 15    # số tin trùng liên tiếp thì dừng
BATCH_FLUSH = 10             # checkpoint xuống đĩa mỗi N tin mới
MIN_DELAY, MAX_DELAY = 1.5, 3.5
UPLOAD_B2 = True             # Tự động đẩy file Parquet lên Backblaze B2 Data Lake sau khi cào


## 3. Thực thi cào dữ liệu (incremental, fault-tolerant, Hive Parquet)

In [ ]:
df_raw, df_filled, stats = crawl_pipeline(
    target_samples=TARGET_SAMPLES,
    output_dir=OUTPUT_DIR,
    state_file=STATE_FILE,
    early_stop_threshold=EARLY_STOP_THRESHOLD,
    min_delay=MIN_DELAY,
    max_delay=MAX_DELAY,
    batch_flush=BATCH_FLUSH,
    log_file=LOG_FILE,
    upload_b2=UPLOAD_B2,
)


## 4. Xem nhanh dữ liệu

In [ ]:
cols = [c for c in ["Mã tin", "Khoảng giá", "Diện tích", "Ngày đăng", "Loại tin", "crawled_at"] if c in df_raw.columns]
(df_raw[cols].tail(10) if not df_raw.empty else "Chưa có dữ liệu (xem crawler.log).")

## 5. Điền khuyết tự động từ mô tả (Regex Imputation)
`crawl_pipeline` đã gọi `fill_specs_with_desc`; bảng dưới so sánh tỷ lệ khuyết trước và sau.

In [ ]:
if df_raw.empty or df_filled.empty:
    result = "Chưa có dữ liệu để so sánh."
else:
    result = pd.DataFrame({
        "Khuyết trước (%)": df_raw.isnull().mean().mul(100).round(1),
        "Khuyết sau điền (%)": df_filled.isnull().mean().mul(100).round(1),
    })
result

## 6. Tổng kết phiên chạy

In [ ]:
pd.Series({
    "Tổng tin quét": stats.total_scanned,
    "Tin mới": stats.new_collected,
    "Tin bỏ qua (trùng)": stats.skipped_duplicates,
    "Tin lỗi": stats.failed_errors,
    "Thời gian (s)": stats.duration_seconds,
    "Lý do dừng": stats.stop_reason,
    "Tổng số mẫu trong đợt này": len(df_raw),
}, name="Tổng kết")

## 7. Backblaze B2 Data Lake: Kiểm tra kết nối & Danh sách file đã lưu
Kiểm tra kết nối tới Bucket Data Lake và xem danh sách các file Parquet phân vùng Hive đã tải lên.

In [ ]:
# Kiểm tra kết nối B2 và liệt kê danh sách file hiện có trong prefix 'raw'
verify_b2_connection_and_list(prefix="raw")


### 7.1 Đồng bộ toàn bộ thư mục Parquet local lên B2 (Idempotent)
Quét thư mục `data_1/Raw/` và tải lên tất cả các file Parquet chưa có trên B2 (bỏ qua file đã tồn tại).

In [ ]:
b2_mgr = get_b2_manager()
sync_result = b2_mgr.sync_raw_directory(raw_dir=OUTPUT_DIR, prefix="raw", overwrite=False)
pd.Series(sync_result, name="B2 Sync Summary")
